# Análisis del afinamiento con RAFT: corpus, entrenamiento y evaluación

Genera, a partir de los archivos ya existentes, todas las cifras del afinamiento con RAFT que cita el documento del trabajo de grado. Cada tabla del documento queda en una hoja de `reporte_raft.xlsx`, y cada cifra tiene un archivo que la respalda. No modifica ningún archivo existente.

| Sección | Qué produce | Hoja del Excel |
|---|---|---|
| 1 | Corpus fuente: fragmentos del PDF, chunks y pares generados y revisados | `Corpus_fuente` |
| 2 | Embudo del corpus RAFT: de los pares revisados a los ejemplos de entrenamiento | `Corpus_embudo` |
| 3 | Composición de los ejemplos RAFT (documento dorado, distractores, longitud) | `Ejemplos_RAFT` |
| 4 | Parámetros y resultado del entrenamiento (requiere las salidas de Kaggle) | `Entrenamiento`, `Perdida` |
| 5 | Métricas de generación: RAFT frente a las combinaciones de Claude (n = 196) | `Metricas_generacion` |
| 6 | Pruebas de Wilcoxon pareadas con corrección de Holm | `Wilcoxon_RAFT` |
| 7 | Comportamiento de las respuestas: fallas, abstenciones, citas y latencia | `Comportamiento` |
| 8 | Desempeño por tipo de evidencia (texto / tabla) | `Por_tipo_evidencia` |

**Archivos que lee:**
- De este proyecto (`./validacion/`): `evaluacion_generacion_detalle.csv`, `evaluacion_generacion_detalle_raft.csv`, `evaluacion_generacion_resumen_raft.csv`, `resultados_generacion_llm.json`, `resultados_generacion_llm_raft.json` y `validacion_respuestas_raft.csv`.
- Del proyecto de corpus (`../TG - Generacion de corpus/`): `JsonsDatasetByChars/dataset_acreditacion_{primera,segunda}_mitad.json` y `raft/salida/` (`cot_cache.jsonl`, `raft_completo.jsonl`, `reporte_raft.json`).
- Las salidas del notebook de Kaggle (`log_entrenamiento.json`, `config_entrenamiento.json`) en `../TG - Generacion de corpus/raft/entrenamiento/` (o en su subcarpeta `kaggle_output/`).
- **Si existen**, los resultados del control con el modelo base: `evaluacion_generacion_detalle_base.csv`, `resultados_generacion_llm_base.json` y `validacion_respuestas_base.csv`. Con ellos se agregan las filas del modelo base y las pruebas RAFT frente a base.

**Salida:** `./validacion/reporte_raft/`, con `reporte_raft.xlsx`, un CSV por tabla y las figuras.

In [ ]:
%pip install -q pandas numpy scipy openpyxl matplotlib

In [ ]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

DIR_VALIDACION = Path("./validacion")
DIR_CORPUS = Path("../TG - Generacion de corpus")
DIR_RAFT = DIR_CORPUS / "raft"
# Salidas de Kaggle: se buscan en entrenamiento/kaggle_output/ o directamente en entrenamiento/
DIR_KAGGLE = next((d for d in [DIR_RAFT / "entrenamiento" / "kaggle_output", DIR_RAFT / "entrenamiento"]
                   if (d / "log_entrenamiento.json").exists()), DIR_RAFT / "entrenamiento" / "kaggle_output")
DIR_SALIDA = DIR_VALIDACION / "reporte_raft"
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

assert DIR_CORPUS.exists(), (f"No encuentro la carpeta del corpus en {DIR_CORPUS.resolve()}. "
                             "Ajusta DIR_CORPUS a la ruta de 'TG - Generacion de corpus'.")

METRICAS = ["faithfulness", "answer_relevancy", "answer_correctness", "semantic_similarity"]
ETIQ_LLM = {"claude-sonnet-5": "Sonnet 5", "claude-haiku-4.5": "Haiku 4.5",
            "raft-llama3.1-8b": "Llama 3.1 8B + RAFT", "llama3.1-8b-base": "Llama 3.1 8B (base)"}
ETIQ_PROMPT = {"directo": "Directo", "estructurado": "Estructurado", "raft": "Formato RAFT"}

tablas = {}   # nombre de hoja -> DataFrame

## 1. Corpus fuente (pares generados por fragmento y revisados a mano)

El documento de 330 páginas se dividió en 22 PDF de 15 páginas. Para cada PDF, `generacion_dataset.py` extrajo el texto por página, separó las tablas, segmentó en chunks y pidió a `meta-llama/llama-3.3-70b-instruct` 3 pares pregunta–respuesta por chunk. Despues, los pares se revisaron a mano: se eliminaban o modificaban los que no eran pertinentes. El resultado consolidado esta en las dos mitades.

In [ ]:
pares_fuente = []
for mitad in ["primera", "segunda"]:
    with open(DIR_CORPUS / "JsonsDatasetByChars" / f"dataset_acreditacion_{mitad}_mitad.json", encoding="utf-8") as f:
        for x in json.load(f):
            x["mitad"] = mitad
            pares_fuente.append(x)
fuente = pd.DataFrame(pares_fuente)
fuente["fragmento_pdf"] = fuente["fuente"].str.extract(r"-(\d+)(?: \(1\))?\.pdf$")[0].astype("Int64")

chunks = fuente.drop_duplicates("contexto")
pares_por_chunk = fuente.groupby("contexto").size()

tabla_fuente = pd.DataFrame({
    "Concepto": [
        "Páginas del documento", "Fragmentos PDF (de hasta 15 páginas)",
        "Modelo generador de pares", "Pares solicitados por chunk",
        "Chunks con al menos un par", "Chunks de texto", "Chunks de tabla",
        "Tokens por chunk (mediana, cl100k)", "Tokens por chunk (máximo)",
        "Pares tras la revisión manual", "Pares de texto", "Pares de tabla",
        "Pares con nota del revisor", "Chunks con 3 pares", "Chunks con menos de 3 pares", "Chunks con más de 3 pares",
    ],
    "Valor": [
        330, 22, "meta-llama/llama-3.3-70b-instruct", 3,
        len(chunks), int((chunks["tipo"] == "texto").sum()), int((chunks["tipo"] == "tabla").sum()),
        float(chunks["token_count"].median()), int(chunks["token_count"].max()),
        len(fuente), int((fuente["tipo"] == "texto").sum()), int((fuente["tipo"] == "tabla").sum()),
        int(fuente["nota"].notna().sum()) if "nota" in fuente else 0,
        int((pares_por_chunk == 3).sum()), int((pares_por_chunk < 3).sum()), int((pares_por_chunk > 3).sum()),
    ],
})
print("Fragmentos PDF presentes:", sorted(fuente["fragmento_pdf"].dropna().unique().tolist()))
tablas["Corpus_fuente"] = tabla_fuente
tabla_fuente

## 2. Embudo del corpus RAFT

Los conteos de limpieza y de exclusion del dataset de validacion vienen de `reporte_raft.json`, que guardo `preparar_corpus_raft.py` al ensamblar. Las etapas con LLM (control de calidad y respuesta con razonamiento) se leen de `cot_cache.jsonl`, y el resultado final de `raft_completo.jsonl`. El notebook comprueba que los tres archivos cuadren entre si.

In [ ]:
import hashlib

with open(DIR_RAFT / "salida" / "reporte_raft.json", encoding="utf-8") as f:
    reporte = json.load(f)
st_limpieza = reporte["limpieza_y_exclusion_validacion"]

# Mismo identificador de chunk que usa preparar_corpus_raft.py (sha1 del texto, 12 caracteres)
chunks_corpus = {hashlib.sha1(t.encode("utf-8")).hexdigest()[:12]: {"tipo": tp}
                 for t, tp in zip(chunks["contexto"], chunks["tipo"])}

cache = {}
with open(DIR_RAFT / "salida" / "cot_cache.jsonl", encoding="utf-8") as f:
    for linea in f:
        if linea.strip():
            r = json.loads(linea)
            cache[r["id"]] = r            # la ultima version gana, igual que en el script
cot = pd.DataFrame(list(cache.values()))
pares_crudos = fuente
pares_limpios = cot                       # lo que se envio al LLM = pares tras la limpieza
ok = cot[cot["estado"] == "ok"]
solapados = ok[ok.get("solapa_evaluacion", pd.Series("NO", index=ok.index)).fillna("NO") != "NO"]

completo = pd.read_json(DIR_RAFT / "salida" / "raft_completo.jsonl", lines=True)

etapas = [
    ("Pares del corpus fuente (revisados a mano)", len(pares_crudos), ""),
    ("Duplicados dentro del mismo chunk", -st_limpieza.get("duplicados_mismo_chunk", 0), "Limpieza"),
    ("Pregunta idéntica a una de validación", -st_limpieza.get("excluidas_pregunta_identica", 0), "Exclusión de validación (léxica)"),
    ("Pregunta similar a una de validación (Jaccard ≥ 0.6)", -st_limpieza.get("excluidas_pregunta_similar", 0), "Exclusión de validación (léxica)"),
    ("Mismo hecho que una evidencia de validación", -st_limpieza.get("excluidas_mismo_hecho_que_validacion", 0), "Exclusión de validación (por evidencia)"),
    ("Pares enviados al LLM", len(pares_limpios), "Subtotal"),
    ("Descartados por el LLM (control de calidad)", -int((cot["estado"] == "descartado").sum()), "Control de calidad"),
    ("Fallidos (citas no literales tras 3 intentos)", -int((cot["estado"] == "fallido").sum()), "Verificación de citas"),
    ("Marcados por el LLM como mismo hecho que validación", -len(solapados), "Exclusión de validación (semántica)"),
    ("Ejemplos RAFT finales", len(completo), "Total"),
    ("  de entrenamiento", int((completo["particion"] == "train").sum()), ""),
    ("  de evaluación (monitoreo de la pérdida)", int((completo["particion"] == "eval").sum()), ""),
]
embudo = pd.DataFrame(etapas, columns=["Etapa", "Pares", "Tipo de filtro"])
tablas["Corpus_embudo"] = embudo

# Control: el recalculo debe coincidir con lo que guardo el script
assert len(pares_limpios) == reporte["pares_de_entrada_tras_limpieza"], "cot_cache no coincide con reporte_raft.json"
assert len(pares_crudos) - sum(st_limpieza.get(k, 0) for k in ("duplicados_mismo_chunk", "excluidas_pregunta_identica",
       "excluidas_pregunta_similar", "excluidas_mismo_hecho_que_validacion")) == len(pares_limpios)
assert len(solapados) == reporte["excluidos_por_solapamiento_llm"]
assert len(completo) == reporte["ejemplos"]["train"] + reporte["ejemplos"]["eval"]
assert len(pares_limpios) - (cot["estado"] != "ok").sum() - len(solapados) == len(completo)
print("Control OK: el embudo cuadra con reporte_raft.json y raft_completo.jsonl")

veredictos = ok.loc[~ok.index.isin(solapados.index), "veredicto"].value_counts()
print("Veredictos de los ejemplos finales:", veredictos.to_dict())
print("Intentos necesarios (pares aceptados):", ok["intentos"].value_counts().sort_index().to_dict())
print("Modelo generador del razonamiento:", cot["modelo"].unique().tolist())
embudo

In [ ]:
detalle_calidad = pd.DataFrame({
    "Concepto": ["Aceptados sin cambios (ACEPTAR)", "Aceptados con pregunta o respuesta ajustada (AJUSTAR)",
                 "Aceptados al primer intento", "Aceptados al segundo intento", "Aceptados al tercer intento",
                 "Citas por razonamiento (media)", "Respuesta final, caracteres (mediana)"],
    "Valor": [int(veredictos.get("ACEPTAR", 0)), int(veredictos.get("AJUSTAR", 0)),
              int((ok["intentos"] == 1).sum()), int((ok["intentos"] == 2).sum()), int((ok["intentos"] == 3).sum()),
              round(ok["razonamiento"].str.count("##begin_quote##").mean(), 2),
              float(ok["respuesta"].str.len().median())],
}).astype({"Valor": object})
tablas["Control_calidad_LLM"] = detalle_calidad
detalle_calidad

## 3. Composición de los ejemplos RAFT

Cada ejemplo tiene la pregunta, 5 documentos y la respuesta con razonamiento. En una fracción P lleva el documento dorado más 4 distractores; en el resto (1 − P) lleva 5 distractores. Se usó P = 0.8 y distractores muestreados al azar del corpus, igual que en el paper.

In [ ]:
completo["n_documentos"] = completo["ids_documentos"].str.len()
tipo_chunk = {k: v["tipo"] for k, v in chunks_corpus.items()}
completo["tipo_dorado"] = completo["id_chunk_dorado"].map(tipo_chunk)

filas = []
for nombre, sub in [("Entrenamiento", completo[completo["particion"] == "train"]),
                    ("Evaluación", completo[completo["particion"] == "eval"]),
                    ("Total", completo)]:
    filas.append({
        "Partición": nombre,
        "Ejemplos": len(sub),
        "Con documento dorado": int(sub["con_documento_dorado"].sum()),
        "Sin documento dorado": int((~sub["con_documento_dorado"]).sum()),
        "Proporción con dorado (P)": round(sub["con_documento_dorado"].mean(), 3),
        "Documento dorado de texto": int((sub["tipo_dorado"] == "texto").sum()),
        "Documento dorado de tabla": int((sub["tipo_dorado"] == "tabla").sum()),
        "Documentos por ejemplo": int(sub["n_documentos"].mode()[0]),
        "Tokens por ejemplo, mediana": int(sub["tokens_aprox"].median()),
        "Tokens por ejemplo, p95": int(sub["tokens_aprox"].quantile(0.95)),
        "Tokens por ejemplo, máximo": int(sub["tokens_aprox"].max()),
    })
ejemplos = pd.DataFrame(filas)
chunks_dorados = completo.groupby("particion")["id_chunk_dorado"].apply(set)
assert not (chunks_dorados["train"] & chunks_dorados["eval"]), "Hay chunks dorados compartidos entre train y eval"
print("Sin chunks dorados compartidos entre entrenamiento y evaluación: OK")
tablas["Ejemplos_RAFT"] = ejemplos
ejemplos.T

## 4. Entrenamiento

Los parámetros vienen de `raft_kaggle_entrenamiento.ipynb`. El tiempo, la GPU y las pérdidas vienen de las salidas de Kaggle (`config_entrenamiento.json` y `log_entrenamiento.json`).

In [ ]:
params = {
    "Modelo base": "Llama 3.1 8B Instruct (unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit)",
    "Método": "QLoRA (pesos base en 4 bits + adaptadores LoRA), Unsloth + TRL SFTTrainer",
    "Módulos LoRA": "q, k, v, o, gate, up, down",
    "Rango r / alpha / dropout": "16 / 32 / 0",
    "Épocas": 2,
    "Tasa de aprendizaje": "2e-4, decaimiento lineal, calentamiento 3 %",
    "Lote efectivo": "16 (2 × acumulación de 8)",
    "Optimizador": "AdamW 8 bits, weight decay 0.01",
    "Longitud máxima de secuencia": 3072,
    "Pérdida calculada sobre": "solo la respuesta del asistente (##Reason … ##Answer)",
    "Plataforma": "Kaggle Notebooks, GPU NVIDIA T4 (16 GB)",
    "Exportación": "modelo fusionado 16 bits y GGUF Q4_K_M (~4.9 GB)",
    "Inferencia en las pruebas": "Ollama local, RTX 4050 Laptop 6 GB, num_ctx 8192, temperature 0",
}

perdida = pd.DataFrame()
if (DIR_KAGGLE / "config_entrenamiento.json").exists():
    cfg = json.loads((DIR_KAGGLE / "config_entrenamiento.json").read_text(encoding="utf-8"))
    params.update({"Ejemplos de entrenamiento / evaluación": f"{cfg.get('n_train')} / {cfg.get('n_eval')}",
                   "GPU (registrada)": cfg.get("gpu"), "Tiempo de entrenamiento (min)": cfg.get("minutos")})
if (DIR_KAGGLE / "log_entrenamiento.json").exists():
    log = json.loads((DIR_KAGGLE / "log_entrenamiento.json").read_text(encoding="utf-8"))
    tr = pd.DataFrame([x for x in log if "loss" in x and "eval_loss" not in x])[["step", "epoch", "loss"]]
    ev = pd.DataFrame([x for x in log if "eval_loss" in x])[["step", "epoch", "eval_loss"]]
    perdida = tr.merge(ev, on=["step", "epoch"], how="outer").sort_values("step")
    resumen_tr = next((x for x in log if "train_loss" in x), {})
    params.update({
        "Pasos de entrenamiento": int(resumen_tr.get("step", tr["step"].max())),
        "Pérdida media de entrenamiento (todas las épocas)": round(resumen_tr["train_loss"], 4) if resumen_tr else None,
        "Pérdida de entrenamiento inicial / final": f"{tr['loss'].iloc[0]:.4f} / {tr['loss'].iloc[-1]:.4f}",
        "Pérdida de evaluación primera / última / mínima": f"{ev['eval_loss'].iloc[0]:.4f} / {ev['eval_loss'].iloc[-1]:.4f} / {ev['eval_loss'].min():.4f}",
    })
    fig, ax = plt.subplots(figsize=(7, 3.4))
    ax.plot(tr["step"], tr["loss"], label="Entrenamiento", color="#2a78d6", lw=1.2)
    ax.plot(ev["step"], ev["eval_loss"], "o-", label="Evaluación", color="#d6612a", lw=1.2, ms=4)
    ax.set_xlabel("Paso"); ax.set_ylabel("Pérdida"); ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(DIR_SALIDA / "perdida_entrenamiento.png", dpi=200); plt.show()
else:
    print(f"AVISO: no encontré las salidas de Kaggle en {DIR_KAGGLE.resolve()}. "
          "La tabla queda solo con los parámetros configurados.")

entrenamiento = pd.DataFrame(list(params.items()), columns=["Parámetro", "Valor"])
tablas["Entrenamiento"] = entrenamiento
if len(perdida):
    tablas["Perdida"] = perdida
entrenamiento

## 5. Métricas de generación (n = 196, contextos de OpenAI denso)

Se ponen juntas las 4 combinaciones de Claude con OpenAI denso (Tabla 13 del documento) y las combinaciones del modelo RAFT. El promedio compuesto se calcula igual que en `analisis_generacion.ipynb`: es la media de las cuatro métricas ya promediadas por combinación.

In [ ]:
cols = ["user_input", "pipeline_retrieve", "modelo_llm", "estilo_prompt"] + METRICAS
claude = pd.read_csv(DIR_VALIDACION / "evaluacion_generacion_detalle.csv", usecols=cols)
claude = claude[claude["pipeline_retrieve"] == "openai_cs800"]
raft = pd.read_csv(DIR_VALIDACION / "evaluacion_generacion_detalle_raft.csv", usecols=cols)
partes = [claude, raft]
RUTA_DET_BASE = DIR_VALIDACION / "evaluacion_generacion_detalle_base.csv"
HAY_BASE = RUTA_DET_BASE.exists()
if HAY_BASE:
    partes.append(pd.read_csv(RUTA_DET_BASE, usecols=cols))
    print("Incluye el control con el modelo base sin afinar.")
else:
    print("AVISO: no hay resultados del modelo base; las tablas solo tienen Claude y RAFT.")
gen = pd.concat(partes, ignore_index=True)
gen["promedio"] = gen[METRICAS].mean(axis=1)          # por respuesta (para Wilcoxon)
assert gen.groupby(["modelo_llm", "estilo_prompt"]).size().eq(196).all(), "Alguna combinación no tiene 196 respuestas"

met = (gen.groupby(["modelo_llm", "estilo_prompt"])[METRICAS].mean().reset_index())
met["Promedio"] = met[METRICAS].mean(axis=1)
met["Nulos"] = gen.groupby(["modelo_llm", "estilo_prompt"])[METRICAS].apply(lambda x: int(x.isna().sum().sum())).values
met["Modelo"] = met["modelo_llm"].map(ETIQ_LLM)
met["Prompt"] = met["estilo_prompt"].map(ETIQ_PROMPT)
met = met.sort_values("Promedio", ascending=False).reset_index(drop=True)
metricas_gen = met[["Modelo", "Prompt"] + METRICAS + ["Promedio", "Nulos", "modelo_llm", "estilo_prompt"]]

# Control contra el resumen guardado por la evaluacion RAFT
res = pd.read_csv(DIR_VALIDACION / "evaluacion_generacion_resumen_raft.csv").set_index("estilo_prompt")
calc = metricas_gen[metricas_gen["modelo_llm"] == "raft-llama3.1-8b"].set_index("estilo_prompt")[METRICAS]
print(f"Diferencia máxima con evaluacion_generacion_resumen_raft.csv: {(res[METRICAS] - calc).abs().max().max():.2e}")
tablas["Metricas_generacion"] = metricas_gen.drop(columns=["modelo_llm", "estilo_prompt"])
metricas_gen.drop(columns=["modelo_llm", "estilo_prompt"]).round(4)

In [ ]:
et = (metricas_gen["Modelo"] + " · " + metricas_gen["Prompt"]).iloc[::-1]
vals = metricas_gen["Promedio"].iloc[::-1]
colores = ["#d6612a" if "RAFT" in m else ("#8a8a8a" if "base" in m else "#2a78d6") for m in metricas_gen["Modelo"].iloc[::-1]]
fig, ax = plt.subplots(figsize=(8, 0.45 * len(vals) + 0.6))
ax.barh(et, vals, color=colores)
for y, v in enumerate(vals):
    ax.text(v + 0.0007, y, f"{v:.4f}", va="center", fontsize=8)
ax.set_xlim(min(0.77, vals.min() - 0.01), max(0.81, vals.max() + 0.01))
ax.set_xlabel("Promedio compuesto (n = 196, OpenAI denso)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(DIR_SALIDA / "promedio_compuesto_raft_vs_claude.png", dpi=200); plt.show()

## 6. Pruebas de Wilcoxon pareadas

Usa las mismas funciones que `pruebas_estadisticas.ipynb`. Cada comparación es una familia de 5 pruebas (el promedio compuesto y las 4 métricas), corregidas con Holm, como en la Tabla 15.

| Comparación | Pregunta que responde |
|---|---|
| RAFT (estructurado) vs. Haiku 4.5 (estructurado) | Con el mismo prompt, ¿el modelo local afinado iguala a la configuración más eficiente? |
| RAFT (estructurado) vs. Sonnet 5 (estructurado) | ¿Y a la configuración de mayor calidad? |
| RAFT (formato RAFT) vs. Haiku 4.5 (estructurado) | El mejor uso previsto de cada modelo frente a frente. |
| RAFT (formato RAFT) vs. Sonnet 5 (estructurado) | Lo mismo, frente a la configuración de mayor calidad. |
| RAFT (formato RAFT) vs. RAFT (estructurado) | ¿Usar en prueba el formato del entrenamiento mejora frente al prompt del proyecto? |
| RAFT vs. Llama 3.1 8B base (si existe) | El control del paper: ¿el efecto es del afinamiento o del modelo? |

In [ ]:
def holm(pvalores):
    p = np.asarray(pvalores, dtype=float)
    orden = np.argsort(p)
    m = len(p)
    ajustado = np.empty(m)
    maximo = 0.0
    for rango, i in enumerate(orden):
        maximo = max(maximo, (m - rango) * p[i])
        ajustado[i] = min(1.0, maximo)
    return ajustado


def wilcoxon_pareado(a, b):
    ab = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
    dif = ab["a"] - ab["b"]
    p = wilcoxon(ab["a"], ab["b"]).pvalue if (dif != 0).any() else 1.0
    return {"n": len(ab), "media A": ab["a"].mean(), "media B": ab["b"].mean(), "Diferencia (A − B)": dif.mean(),
            "p": p, "Gana A": int((dif > 0).sum()), "Gana B": int((dif < 0).sum()), "Empates": int((dif == 0).sum())}


def serie(modelo, prompt, columna):
    d = gen[(gen["modelo_llm"] == modelo) & (gen["estilo_prompt"] == prompt)]
    assert len(d), f"No hay respuestas para {modelo} / {prompt}"
    return d.set_index("user_input")[columna]


R, H, S = "raft-llama3.1-8b", "claude-haiku-4.5", "claude-sonnet-5"
COMPARACIONES = [
    ((R, "estructurado"), (H, "estructurado")),
    ((R, "estructurado"), (S, "estructurado")),
    ((R, "raft"), (H, "estructurado")),
    ((R, "raft"), (S, "estructurado")),
    ((R, "raft"), (R, "estructurado")),
]
if "llama3.1-8b-base" in set(gen["modelo_llm"]):
    for p in sorted(set(gen.loc[gen["modelo_llm"] == "llama3.1-8b-base", "estilo_prompt"])):
        COMPARACIONES.append(((R, p), ("llama3.1-8b-base", p)))

filas = []
for (ma, pa), (mb, pb) in COMPARACIONES:
    nombre = f"{ETIQ_LLM[ma]} ({ETIQ_PROMPT[pa]}) vs. {ETIQ_LLM[mb]} ({ETIQ_PROMPT[pb]})"
    bloque = [{"Comparación": nombre, "Métrica": m, **wilcoxon_pareado(serie(ma, pa, m), serie(mb, pb, m))}
              for m in ["promedio"] + METRICAS]
    ph = holm([b["p"] for b in bloque])
    for b, x in zip(bloque, ph):
        b["p Holm (5 métricas)"] = x
        b["Resultado"] = ("Empate" if x >= 0.05 else ("Gana A" if b["Diferencia (A − B)"] > 0 else "Gana B"))
    filas += bloque
tabla_w = pd.DataFrame(filas)
tablas["Wilcoxon_RAFT"] = tabla_w
pd.set_option("display.max_colwidth", 90)
tabla_w.round(4)

## 7. Comportamiento de las respuestas

- **Fallas del modelo:** las mismas reglas de `evaluacion_generacion_ragas_raft.ipynb` (bucles, respuesta cortada, formato no respetado).
- **Abstenciones:** se detectan con el mismo patrón para todos los modelos.
- **Citas literales:** solo aplica al formato RAFT.
- **Latencia:** solo para el modelo local. En Claude no se midió.

In [ ]:
PATRON_ABSTENCION = re.compile(
    r"no se encuentra|no (contiene|incluye|proporciona|menciona|especifica)|no hay informaci|no es posible (determinar|responder)",
    re.IGNORECASE)

with open(DIR_VALIDACION / "resultados_generacion_llm.json", encoding="utf-8") as f:
    gen_claude = [c for c in json.load(f)["combinaciones"] if c["pipeline_retrieve"] == "openai_cs800"]
with open(DIR_VALIDACION / "resultados_generacion_llm_raft.json", encoding="utf-8") as f:
    gen_local = json.load(f)["combinaciones"]
vals_csv = [pd.read_csv(DIR_VALIDACION / "validacion_respuestas_raft.csv")]
if HAY_BASE:
    with open(DIR_VALIDACION / "resultados_generacion_llm_base.json", encoding="utf-8") as f:
        gen_local += json.load(f)["combinaciones"]
    vals_csv.append(pd.read_csv(DIR_VALIDACION / "validacion_respuestas_base.csv"))

val = pd.concat(vals_csv, ignore_index=True)
fallas = val.groupby(["modelo_llm", "estilo_prompt"])["falla_modelo"].sum()
citas = val[val["estilo_prompt"] == "raft"].groupby("modelo_llm")[["info_citas", "info_citas_literales"]].sum()

filas = []
for c in gen_claude + gen_local:
    rs = c["resultados"]
    resp = [r.get("respuesta_generada") or "" for r in rs]
    lat = pd.Series([r.get("latencia_s") for r in rs], dtype="float")
    local = c["modelo_llm"] in ("raft-llama3.1-8b", "llama3.1-8b-base")
    fila = {
        "Modelo": ETIQ_LLM.get(c["modelo_llm"], c["modelo_llm"]),
        "Prompt": ETIQ_PROMPT[c["estilo_prompt"]],
        "Respuestas": len(rs),
        "Abstenciones": sum(bool(PATRON_ABSTENCION.search(x)) for x in resp),
        "Palabras por respuesta (mediana)": float(np.median([len(x.split()) for x in resp])),
        "Fallas del modelo": int(fallas.get((c["modelo_llm"], c["estilo_prompt"]), 0)),
        "Formato pedido respetado": (sum("##Answer:" in (r.get("respuesta_generada_cruda") or "") for r in rs) if c["estilo_prompt"] == "raft"
                                     else sum("Respuesta final:" in (r.get("respuesta_generada_cruda") or "") for r in rs) if c["estilo_prompt"] == "estructurado"
                                     else None),
        "Citas literales / citas": (f"{int(citas.loc[c['modelo_llm'], 'info_citas_literales'])} / {int(citas.loc[c['modelo_llm'], 'info_citas'])}"
                                    if c["estilo_prompt"] == "raft" and c["modelo_llm"] in citas.index else "—"),
        "Latencia mediana (s)": round(lat.median(), 1) if local else None,
        "Latencia p95 (s)": round(lat.quantile(0.95), 1) if local else None,
        "Tokens de entrada (mediana)": int(np.median([r.get("tokens_prompt") or 0 for r in rs])) if local else None,
        "Tokens de entrada (máximo)": int(max(r.get("tokens_prompt") or 0 for r in rs)) if local else None,
    }
    filas.append(fila)
comportamiento = pd.DataFrame(filas)
tablas["Comportamiento"] = comportamiento
comportamiento

## 8. Desempeño por tipo de evidencia

El tipo de evidencia de cada pregunta (texto o tabla) se toma del dataset de validación. Las 4 preguntas de "texto y tabla" se agrupan con tabla. El promedio compuesto es por respuesta.

In [ ]:
tipo_ev = {r["pregunta"]: r.get("tipo_evidencia") for c in gen_local for r in c["resultados"]}
gen["tipo_evidencia"] = gen["user_input"].map(tipo_ev).replace({"texto+tabla": "tabla"})
assert gen["tipo_evidencia"].notna().all()

por_tipo = (gen.groupby(["modelo_llm", "estilo_prompt", "tipo_evidencia"])[["promedio", "faithfulness", "answer_correctness"]]
               .mean().unstack("tipo_evidencia"))
por_tipo.columns = [f"{m} – {t}" for m, t in por_tipo.columns]
por_tipo["n texto / tabla"] = (gen.groupby(["modelo_llm", "estilo_prompt", "tipo_evidencia"]).size()
                               .unstack().apply(lambda r: f"{r['texto']} / {r['tabla']}", axis=1))
por_tipo = por_tipo.reset_index()
por_tipo.insert(0, "Prompt", por_tipo.pop("estilo_prompt").map(ETIQ_PROMPT))
por_tipo.insert(0, "Modelo", por_tipo.pop("modelo_llm").map(ETIQ_LLM))
por_tipo = por_tipo.sort_values("promedio – tabla", ascending=False)
tablas["Por_tipo_evidencia"] = por_tipo
por_tipo.round(4)

## 9. Exportación

In [ ]:
with pd.ExcelWriter(DIR_SALIDA / "reporte_raft.xlsx") as xw:
    for hoja, t in tablas.items():
        t.to_excel(xw, sheet_name=hoja[:31], index=False)
for hoja, t in tablas.items():
    t.to_csv(DIR_SALIDA / f"{hoja.lower()}.csv", index=False, encoding="utf-8-sig")
print("Hojas:", list(tablas))
print("Guardado en", DIR_SALIDA.resolve())